# Comparaison EmbeddingGemma / COMETKiwi

Notebook séparé parce que unbabel-comet impose des versions plus anciennes de certaines bibliothèques, ce qui cassait le gros notebook. Ici on l'installe en premier dans un Colab neuf.

Il relit resultats_conclusion_embeddinggemma.xlsx (phrases, traductions et notes d'EmbeddingGemma), fait noter les mêmes traductions par COMETKiwi, puis compare les deux.

Ordre : lancer l'installation, redémarrer la session (Exécution → Redémarrer la session), puis lancer le reste dans l'ordre en choisissant le fichier xlsx quand il est demandé. À la fin un fichier Excel est téléchargé. La cellule de chargement écrit aussi a_noter_avec_cometkiwi.csv, au cas où COMETKiwi ne s'installe pas et qu'il faille le faire noter par quelqu'un d'autre.

Il faut avoir accepté la licence de Unbabel/wmt22-cometkiwi-da sur Hugging Face et avoir le secret HF_TOKEN dans Colab.

In [ ]:
# installation, puis redémarrer la session
!pip install -q unbabel-comet scikit-learn scipy openpyxl pandas

Après le redémarrage, ne pas relancer l'installation.

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
IN_COLAB = "google.colab" in sys.modules
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    login()

In [ ]:
# fichier de résultats
PATH = "resultats_conclusion_embeddinggemma.xlsx"
if not os.path.exists(PATH) and IN_COLAB:
    from google.colab import files
    PATH = list(files.upload().keys())[0]

d = pd.read_excel(PATH, sheet_name="donnees")
SYSTEMS = [c[5:] for c in d.columns if c.startswith("hyp::")]
need = ["ar"] + [f"{p}::{s}" for s in SYSTEMS for p in ("hyp", "cos", "chrF")]
missing = [c for c in need if c not in d.columns]
assert not missing, f"Colonnes absentes dans le fichier : {missing}"
print(len(d), "phrases | systèmes :", SYSTEMS)

# au cas où comet ne s'installe pas
rows = [{"systeme": s, "id": i, "src": d.ar[i], "mt": d[f"hyp::{s}"][i]} for s in SYSTEMS for i in range(len(d))]
pd.DataFrame(rows).to_csv("a_noter_avec_cometkiwi.csv", index=False)
print("Fichier écrit : a_noter_avec_cometkiwi.csv")

In [ ]:
# notation avec COMETKiwi
import torch
from comet import download_model, load_from_checkpoint

kiwi = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))
gpus = 1 if torch.cuda.is_available() else 0

KIWI, KIWI_TIME = {}, {}
for s in SYSTEMS:
    t0 = time.perf_counter()
    out = kiwi.predict([{"src": a, "mt": m} for a, m in zip(d.ar, d[f"hyp::{s}"])], batch_size=16, gpus=gpus)
    KIWI_TIME[s] = time.perf_counter() - t0
    KIWI[s] = np.array(out.scores)
    print(s, ": noté en", round(KIWI_TIME[s], 1), "secondes")

In [ ]:
def eval_scorer(score, chrf):
    # score : note du contrôleur, chrf : qualité de chaque phrase
    lo, hi = np.quantile(chrf, 0.25), np.quantile(chrf, 0.75)
    bad, good = score[chrf <= lo], score[chrf >= hi]
    auc = roc_auc_score(np.r_[np.ones(len(good)), np.zeros(len(bad))], np.r_[good, bad])
    keep = score > np.quantile(score, 0.20)
    return {"Spearman": round(float(spearmanr(score, chrf)[0]), 2),
            "AUC": round(float(auc), 2),
            "chrF++ après filtre 20%": round(float(chrf[keep].mean()), 1)}

def zs(x): return (x - x.mean()) / x.std()

rows = []
for s in SYSTEMS + ["tous"]:
    if s == "tous":
        chrf = np.concatenate([d[f"chrF::{x}"].values for x in SYSTEMS])
        cos = np.concatenate([d[f"cos::{x}"].values for x in SYSTEMS]); kw = np.concatenate([KIWI[x] for x in SYSTEMS])
    else:
        chrf, cos, kw = d[f"chrF::{s}"].values, d[f"cos::{s}"].values, KIWI[s]
    combo = zs(cos) + zs(kw)
    ideal = chrf[chrf > np.quantile(chrf, 0.20)].mean()
    for name, sc in [("EmbeddingGemma", cos), ("COMETKiwi", kw), ("combiné", combo)]:
        r = {"système": s, "contrôleur": name, **eval_scorer(sc, chrf)}
        r["chrF++ sans filtre"] = round(float(chrf.mean()), 1); r["chrF++ filtre idéal"] = round(float(ideal), 1)
        rows.append(r)
res = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
display(res)

ov = []
for s in SYSTEMS:
    e, k = d[f"cos::{s}"].values, KIWI[s]
    we, wk = set(np.argsort(e)[:int(0.2 * len(e))]), set(np.argsort(k)[:int(0.2 * len(k))])
    ov.append({"système": s, "recouvrement des 20% rejetés": round(len(we & wk) / len(we), 2),
               "Spearman(EmbeddingGemma, COMETKiwi)": round(float(spearmanr(e, k)[0]), 2),
               "temps COMETKiwi (s)": round(KIWI_TIME[s], 1)})
res_ov = pd.DataFrame(ov); display(res_ov)

In [ ]:
with pd.ExcelWriter("resultats_cometkiwi_vs_embeddinggemma.xlsx") as w:
    res.to_excel(w, sheet_name="comparaison", index=False)
    res_ov.to_excel(w, sheet_name="accord", index=False)
    out = d.copy()
    for s in SYSTEMS: out[f"kiwi::{s}"] = KIWI[s]
    out.to_excel(w, sheet_name="donnees", index=False)
print("Écrit : resultats_cometkiwi_vs_embeddinggemma.xlsx")
if IN_COLAB:
    from google.colab import files
    files.download("resultats_cometkiwi_vs_embeddinggemma.xlsx")